# Flower Classification Using Convolutional Neural Network (CNN)

This project classifies flower images into five categories using a Convolutional Neural Network (CNN).

**Dataset:** TensorFlow flower photos dataset  
**Classes:** daisy, dandelion, roses, sunflowers, tulips  
**Framework:** TensorFlow / Keras


## 1. Import libraries

In [ ]:
import os
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

print("TensorFlow version:", tf.__version__)


## 2. Download and extract the dataset

In [ ]:
dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
archive_path = tf.keras.utils.get_file(
    origin=dataset_url,
    fname="flower_photos.tgz",
    untar=True
)
data_dir = pathlib.Path(archive_path).with_name("flower_photos")

print("Dataset directory:", data_dir)
print("Class folders:", sorted([p.name for p in data_dir.iterdir() if p.is_dir()]))


## 3. Load and prepare the images

In [ ]:
batch_size = 32
img_height = 180
img_width = 180
seed = 123

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=seed,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=seed,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Flower classes:", class_names)


## 4. Display sample images

In [ ]:
plt.figure(figsize=(10, 7))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")
plt.tight_layout()
plt.show()


## 5. Optimize the input pipeline

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)


## 6. Build the CNN model

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

model = models.Sequential([
    layers.Input(shape=(img_height, img_width, 3)),
    data_augmentation,
    layers.Rescaling(1.0 / 255),
    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


## 7. Train the CNN

In [ ]:
epochs = 10

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs
)


## 8. Evaluate the model

In [ ]:
val_loss, val_accuracy = model.evaluate(val_ds, verbose=0)
print(f"Validation loss: {val_loss:.4f}")
print(f"Validation accuracy: {val_accuracy * 100:.2f}%")


## 9. Visualize training performance

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("CNN Flower Classification Accuracy")
plt.legend()
plt.grid(True)
plt.show()


## 10. Test image predictions

In [ ]:
for images, labels in val_ds.take(1):
    probabilities = model.predict(images, verbose=0)
    predicted_labels = np.argmax(probabilities, axis=1)

    plt.figure(figsize=(12, 8))
    for i in range(min(9, len(images))):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        predicted_name = class_names[predicted_labels[i]]
        actual_name = class_names[int(labels[i])]
        color = "green" if predicted_labels[i] == int(labels[i]) else "red"
        plt.title(f"Pred: {predicted_name}\nTrue: {actual_name}", color=color)
        plt.axis("off")
    plt.tight_layout()
    plt.show()


## 11. Save the trained model

In [ ]:
model.save("flower_classification_cnn.keras")
print("Model saved as flower_classification_cnn.keras")


## Conclusion

This notebook demonstrates an image-classification workflow using a CNN. The model is trained to recognize five flower categories: daisy, dandelion, roses, sunflowers, and tulips. The validation accuracy and prediction examples are produced when the notebook is executed.

**Note:** Training results can vary depending on the runtime, dataset split, and training conditions.
